# Demo 4 — Front-loading and reusable preambles
<a id="top"></a>

```yaml
title: Demo 4 — Front-loading and reusable preambles
keywords: front-loading, preamble, system message, overhead, input tokens, context
estimated duration: 12
```

> **Lesson L01**, teacher demo 4. Written lecture [L0102_lecture.md](L0102_lecture.md) §6.
>
> ⚠️ **This whole notebook is live**: it calls the Anthropic API and needs
> `ANTHROPIC_API_KEY`. It shows the lever *you* control — the context you put in front — and how
> reusing it becomes overhead you carry on every call.

**Contents**

1. [Bare vs front-loaded](#1-bare-vs-front-loaded)
2. [The preamble is overhead](#2-the-preamble-is-overhead)
3. [Hand-off to budgeting](#3-hand-off-to-budgeting)

In [ ]:
from __future__ import annotations

import tiktoken

from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

client: PotatoLLMClient = AnthropicClient()  # needs ANTHROPIC_API_KEY
ENC = tiktoken.get_encoding("cl100k_base")  # offline, just to count the preamble

# A reusable block of instruction/persona/format — an informal system prompt (L02 formalizes it).
PREAMBLE = (
    "You are a release-notes editor. Rewrite the input as exactly three past-tense bullet points, "
    "each under 12 words, plain language, no marketing adjectives. Output only the bullets."
)
PASSAGE = (
    "We shipped a big update: login is faster now, we fixed the crash when uploading big files, "
    "and dark mode finally works on the settings screen."
)

## 1. Bare vs front-loaded

Same model, same passage — the front-loaded context (role + format) does the work.

In [ ]:
bare = await client.achat([Message.user(f"Summarize this.\n\n{PASSAGE}")], max_tokens=200)
front = await client.achat([Message.system(PREAMBLE), Message.user(PASSAGE)], max_tokens=200)
print("--- bare ---\n", bare.text)
print("\n--- front-loaded ---\n", front.text)

**A preamble is a reusable, swappable block.** Hold the passage fixed and change *only* the
front-loaded context: the same input comes back shaped a different way each time. That is why we
call it a *preamble* — a lever you write once and reuse, or swap, per job. Here are three more on
the same passage.

In [ ]:
# Same PASSAGE as above — only the front-loaded preamble changes.
MORE_PREAMBLES = {
    "one-line changelog": (
        "You are a changelog bot. Compress the input into a single git-style line: an "
        "imperative summary under 60 characters. Output only that line."
    ),
    "customer announcement": (
        "You are a support lead. Turn the input into a warm two-sentence announcement for "
        "customers. Friendly and concrete, no jargon."
    ),
    "risk review": (
        "You are a cautious release manager. List any risks or follow-ups the input implies, "
        "as short bullets. If there are none, reply 'No risks noted.'"
    ),
}

for name, preamble in MORE_PREAMBLES.items():
    reply = await client.achat([Message.system(preamble), Message.user(PASSAGE)], max_tokens=120)
    print(f"--- {name} ---\n{reply.text}\n")

## 2. The preamble is overhead

The preamble that made the output good is **re-sent on every call**. Count it once (offline),
then watch it ride along in the input tokens of three separate calls.

In [ ]:
print(f"preamble alone: {len(ENC.encode(PREAMBLE))} tokens — re-sent on EVERY call\n")

PASSAGES = [
    "The search box now supports filters.",
    "We removed the old export button and added CSV export.",
    "Notifications can be muted per channel.",
]
for p in PASSAGES:
    reply = await client.achat([Message.system(PREAMBLE), Message.user(p)], max_tokens=120)
    print(f"input_tokens={reply.usage.input_tokens:>4}  (preamble + '{p[:32]}...')")

## 3. Hand-off to budgeting

Because there is **no server-side memory**, that preamble is re-read, re-counted against the
window, and re-billed every single call. You just built up overhead *on purpose* to get better
output. Demo 5 is the bill — in both space (the window) and money (cost).

[↑ Back to top](#top)